# Hopfield 1982：论文复现


**a 储存输入 → b 权重矩阵 → c 检索输入初态 → d 动力学检索 → e 测量指标 → f 展示**




## 0. 论文实验说明


| 类别 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| a. 储存输入 | `a1` 随机记忆 | `a2` 指定汉明距离<br>的相似记忆 | `a3` 围绕共同原型<br>生成相关记忆 |  |  |  |
| b. 权重矩阵 | `b1` 标准 Hebb | `b2` 随机非对称 | `b3` 截断为符号 | `b4` 单向权重 | `b5` 饱和权重 | `b6` 加入序列项 |
| c. 检索输入初态 | `c1` 从指定记忆开始 | `c2` 随机状态 | `c3` 记忆加指定数量扰动 | `c4` 陌生状态 |  |  |
| d. 动力学检索 | `d1` 异步更新 | `d2` 同步更新 |  |  |  |  |
| e. 测量指标 | `e1` 汉明距离 | `e2` 最近吸引子 | `e3` 有效状态数 | `e4` 初始翻转率 | `e5` 最近记忆路径 | `e6` 信号与噪声 |
| f. 展示 | `f1` 曲线图 | `f2` 柱状图 |  |  |  |  |
---
|实验|a|b|c|d|e / f|实验结论|
|---|---|---|---|---|---|---|
|1. 非对称动力学|跳过|`b2` 随机非对称|`c2` 随机|`d1`|`e3 + f2`|非对称网络不保证收敛；固定点与复杂持续动力学可以共存。|
|2. 存储容量|`a1`|`b1`|`c1`|`d1`|`e1 + f1`|随负载增加，召回从稳定逐渐恶化；约在 $0.15N$ 附近出现明显容量压力。|
|3. 随机初态落点|`a1`|`b1`|`c2`|`d1`|`e2 + f2`|只有精确命中才计入已存记忆；原始记忆与反相记忆合计后与论文的名义稳定状态比较。|
|4. 吸引盆纠错|`a1`|`b1`|`c3` 扰动|`d1`|`e2 + f1`|小扰动可以稳定纠正；距离增大后召回率下降。|
|5. 截断权重|`a1`|`b1 → b3`|`c1`|`d1`|`e1 + f2`|截断权重与普通权重在低负载下表现接近，高负载下更快恶化。|
|机制扩展 1：自然遗忘|`a1`|`b5` 饱和权重|`c1`|`d1`|`e1 + f1`|权重饱和使后写入记忆逐渐覆盖早期贡献，形成与记忆年龄相关的遗忘。|
|6. 单向连接|`a1`|`b1 → b4`|`c1`|`d1`|`e1 + e6 + f2`|删除一半方向后性能下降，但网络没有完全失效。|
|7. 相似记忆融合|`a2`|`b1`|`c1`|`d1`|`e1 + e2 + f1`|记忆越接近，吸引子越容易偏移和融合。|
|机制扩展 2：阈值识别新颖性|`a1`|`b1`|`c1` 对比 `c4`|`d1` + 阈值|`f1`|提高阈值可使熟悉与陌生输入呈现不同的静默率。|
|8. 过载熟悉度|`a1`,n=500|`b1`|`c1` 对比 `c4`|前 $N/2$ 次异步尝试|`e4 + 配对统计 + 组合图`|逐项对照论文条件、稳定性和方向；论文未发表可直接复刻的均值。|
|机制扩展 3：相关记忆补全|`a3`|`b1`|`c3` 扰动|`d1`|`e1 + f2`|网络可利用相关记忆的共同原型结构补全输入。|
|9. 时间序列|`a1`|`b1 → b6`|`c1`|`d1`|`e5 + f2`|非对称序列项可推动状态离开当前记忆区域；较长序列是否稳定取决于实际轨迹。|
|10. 同步 vs 异步|`a1`|`b1` 对称权重|`c2` 同一随机初态|`d1` 对比 `d2`|终态类型 + 能量轨迹 + 组合图|异步逐点更新收敛到固定点；同步整体更新可能进入二周期。|



- numpy 创建数组  
  - `np.array(list) --> ndarray`  
    - 将列表转换为 numpy 数组  
    - eg: `np.array([-1,1]) --> array([-1, 1])`  
  - `np.zeros((N,N)) --> ndarray(N,N)`  
    - 创建指定形状的全零矩阵（此处为 N×N）  
    - eg: `np.zeros((2,2)) --> [[0,0],[0,0]]`  
  - `np.zeros_like(权重) --> ndarray`  
    - 创建与给定数组形状相同的全零矩阵  
    - eg: 权重是(3,3) --> 返回一个同样(3,3)的全零矩阵  
  - `np.ones(N) --> ndarray(N,)`  
    - 创建长度为 N 的全一向量  
    - eg: `np.ones(3) --> [1,1,1]`  
  - `np.arange(a,b) --> ndarray(b-a,)`  
    - 生成从 a 到 b-1 的整数序列（左闭右开）  
    - eg: `np.arange(1,5) --> [1,2,3,4]`  
  - `np.asarray(list) --> ndarray`  
    - 将输入转换为 ndarray（若已是数组则不变）  
    - eg: `np.asarray([1,2,3]) --> array([1,2,3])`

- 变形/组合  
  - `np.vstack([a,b]) --> ndarray`  
    - 垂直堆叠两个数组，行数相加  
    - eg: 记忆是(5,30)，`np.vstack([记忆,-记忆])` --> 变成(10,30)的矩阵，上半是原记忆下半是取反  
  - `np.repeat(原型, n, axis=0) --> ndarray(n,N)`  
    - 沿指定轴重复数组元素（此处沿行方向复制 n 次）  
    - eg: 原型是长度30的一维向量，复制5次 --> (5,30)的矩阵，五行完全一样  
  - `np.outer(a,b) --> ndarray(len(a),len(b))`  
    - 计算两个向量的外积  
    - eg: `np.outer([1,-1],[1,1]) --> [[1,1],[-1,-1]]`

- 统计/规约  
  - `np.sum(arr) --> 标量`  
    - 计算数组所有元素的和  
    - eg: `np.sum([1,2,3]) --> 6`  
  - `np.mean(arr) --> 标量`  
    - 计算数组所有元素的平均值  
    - eg: `np.mean([0,2,4]) --> 2.0`  
  - `np.count_nonzero(条件) --> 标量(int)`  
    - 统计数组中满足条件（非零）的元素个数，通常用于布尔数组统计 True 的个数  
    - eg: `a=[1,-1,1], b=[1,1,1]`，`np.count_nonzero(a!=b) --> 1`（只有中间那位不同）  
  - `np.diff(arr) --> ndarray`  
    - 计算相邻元素的差值，结果长度减 1  
    - eg: `np.diff([5,3,3,1]) --> [-2,0,-2]`  
  - `np.diag(权重) --> ndarray(N,)`  
    - 提取方阵的主对角线元素（若输入为矩阵）  
    - eg: 权重是(3,3) --> 取出主对角线上那3个数，变成长度3的一维数组  
  - `np.unique(arr, return_counts=True) --> (去重值, 各自计数)`  
    - 返回数组中的唯一值及其出现次数（若设置 return_counts）  
    - eg: `np.unique(['fixed','fixed','max_sweeps']) --> (['fixed','max_sweeps'], [2,1])`  
  - `np.argmin(arr) --> 标量(索引)`  
    - 返回最小值所在位置的索引（平坦化后的索引）  
    - eg: `np.argmin([5,1,3]) --> 1`（最小值1在下标1）  
  - `np.argsort(arr) --> ndarray`  
    - 返回排序后元素在原数组中的索引（从小到大）  
    - eg: `np.argsort([3,1,2]) --> [1,2,0]`（从小到大排，原来下标1的最小）

- 逐元素变换  
  - `np.sign(arr) --> ndarray`  
    - 返回每个元素的符号（正为 1，零为 0，负为 -1）  
    - eg: `np.sign([-2.5, 0, 3.1]) --> [-1., 0., 1.]`  
  - `np.clip(arr, 下限, 上限) --> ndarray`  
    - 将元素限制在给定区间内，超出部分截断  
    - eg: `np.clip([-5,0,5], -3,3) --> [-3,0,3]`  
  - `np.log(arr) --> ndarray`  
    - 计算每个元素的自然对数  
    - eg: `np.log([1, 2.718...]) --> [0., 1.]`

- 判断/比较  
  - `np.allclose(a,b) --> bool`  
    - 判断两个数组是否在容差范围内近似相等  
    - eg: `np.allclose([1.0000001],[1.0]) --> True`  
  - `np.array_equal(a,b) --> bool`  
    - 判断两个数组是否完全相等（形状和元素）  
    - eg: `np.array_equal([1,2],[1,2]) --> True`  
  - `np.all(arr) --> bool`  
    - 判断数组所有元素是否均为 True（或非零）  
    - eg: `np.all([True,True,False]) --> False`  
  - `np.isscalar(x) --> bool`  
    - 判断输入是否为 Python 标量（不是数组或列表）  
    - eg: `np.isscalar(5.0) --> True`；`np.isscalar([5.0]) --> False`

- 原地修改  
  - `np.fill_diagonal(权重, 值) --> None`  
    - 将方阵的主对角线全部赋为指定值，直接修改原数组，无返回值  
    - eg: 权重原本是 `[[9,1],[1,9]]`，调用 `np.fill_diagonal(权重,0)` 后权重变成 `[[0,1],[1,0]]`，没有返回值可接

- 类型标注（非函数调用）  
  - `np.int8`  
    - 一种数据类型标签，表示数组元素为 8 位有符号整数，常在 dtype 参数中使用  
    - eg: 无具体示例，用于类型声明  
  - `np.ndarray`  
    - numpy 数组的类型本身，用于类型注解中标注变量为 numpy 数组  
    - eg: 无具体示例，用于类型注解

- 随机数生成器  
  - `np.random.default_rng(种子) --> Generator对象`  
    - 创建一个可复现的随机数生成器，种子为整数（可选）  
    - eg: `np.random.default_rng(0) --> <一个可复现的随机数生成器>`  
  - `rng.choice(候选, size=形状) --> ndarray(形状)`  
    - 从候选中随机抽取元素，生成指定形状的数组  
    - eg: `rng.choice([-1,1], size=(2,3))` --> 一个2行3列、每格是-1或+1的矩阵（具体数字随种子变）  
  - `rng.integers(n) --> 标量(int)`  
    - 生成 0 到 n-1 之间的随机整数  
    - eg: `rng.integers(5) --> 一个0到4之间的整数，比如3`  
  - `rng.permutation(N) --> ndarray(N,)`  
    - 返回 0 到 N-1 的随机乱序排列  
    - eg: `rng.permutation(4) --> [2,0,3,1]`  
  - `rng.random((n,N)) --> ndarray(n,N)`  
    - 生成形状为 (n,N) 的均匀分布随机小数（0~1）  
    - eg: 生成一个2行2列、每格是0~1之间随机小数的矩阵，用于跟 flip_probability 比大小  
  - `rng.uniform(-1,1,size=(N,N)) --> ndarray(N,N)`  
    - 生成形状为 (N,N) 的均匀分布随机数，范围在 -1 到 1 之间  
    - eg: 生成一个N×N、每格在-1到1之间均匀分布的矩阵，就是 b2 的权重

- matplotlib 绘图  
  - `plt.style.use(风格名) --> None`  
    - 应用 matplotlib 绘图风格，全局副作用  
    - eg: 无具体示例，直接调用如 `plt.style.use('seaborn')`  
  - `plt.subplots(figsize=(宽,高)) --> (Figure对象, Axes对象)`  
    - 创建画布和坐标系，返回 Figure 和 Axes 对象  
    - eg: `fig, ax = plt.subplots(figsize=(7,4))` --> fig 是整张画布，ax 是画布上的一个坐标系  
  - `plt.show() --> None`  
    - 显示当前所有图形，渲染到屏幕  
    - eg: 无参数，直接调用  
  - `plt.tight_layout() --> None`  
    - 自动调整子图间距，防止重叠  
    - eg: 无参数，直接调用  
  - `plt.xticks(rotation=角度) --> None`  
    - 设置 x 轴刻度标签的旋转角度  
    - eg: `plt.xticks(rotation=45)`  
  - `ax.plot(x,y) --> None`  
    - 在指定坐标系上绘制折线图  
    - eg: `ax.plot([1,2],[3,4])` 画一条线  
  - `ax.bar(标签们, 数值们) --> None`  
    - 绘制柱状图  
    - eg: `ax.bar(['A','B'],[5,7])`  
  - `ax.set(xlabel=.., ylabel=.., title=..) --> None`  
    - 一次设置 x 轴标签、y 轴标签和标题  
    - eg: `ax.set(xlabel='x', ylabel='y', title='my plot')`  
  - `ax.set_ylabel(文字) --> None`  
    - 设置 y 轴标签  
    - eg: `ax.set_ylabel('accuracy')`  
  - `ax.set_title(文字) --> None`  
    - 设置子图标题  
    - eg: `ax.set_title('training loss')`  
  - `ax.tick_params(axis="x", labelrotation=0) --> None`  
    - 设置刻度参数，如旋转 x 轴标签  
    - eg: `ax.tick_params(axis="x", labelrotation=90)`  
  - `ax.legend() --> None`  
    - 显示图例（需先设置 label）  
    - eg: `ax.legend(loc='best')`

- math 模块  
  - `erfc(x) --> 标量(float)`  
    - 计算互补误差函数值  
    - eg: `erfc(0) --> 1.0`；`erfc(2) --> 约0.0047`  
  - `exp(x) --> 标量(float)`  
    - 计算自然指数 e^x  
    - eg: `exp(1) --> 约2.718`  
  - `sqrt(x) --> 标量(float)`  
    - 计算平方根  
    - eg: `sqrt(9) --> 3.0`

- typing / dataclasses（类型层面，无运行时输入输出）  
  - `Optional[X]`  
    - 类型注解，表示该参数可以是 X 类型或 None  
    - eg: 在函数定义中 `def f(x: Optional[int]) -> None:`  
  - `@dataclass`  
    - 装饰器，将类自动添加 `__init__`、`__repr__` 等方法  
    - eg: `@dataclass class Point: x: int; y: int`

- Python 内置函数  
  - `int(x) --> int`  
    - 将数值或字符串转换为整数（截断小数）  
    - eg: `int(3.7) --> 3`  
  - `float(x) --> float`  
    - 将数值或字符串转换为浮点数  
    - eg: `float('3.14') --> 3.14`  
  - `str(x) --> str`  
    - 将对象转换为字符串  
    - eg: `str(123) --> '123'`  
  - `tuple(可迭代对象) --> tuple`  
    - 将可迭代对象转换为元组  
    - eg: `tuple([1,2,3]) --> (1,2,3)`  
  - `list(可迭代对象) --> list`  
    - 将可迭代对象转换为列表  
    - eg: `list(range(3)) --> [0,1,2]`  
  - `dict() --> dict`  
    - 创建空字典或从键值对构造字典  
    - eg: `dict(a=1, b=2) --> {'a':1, 'b':2}`  
  - `set(可迭代对象) --> set`  
    - 将可迭代对象转换为集合（去重）  
    - eg: `set([1,1,2]) --> {1,2}`  
  - `len(容器) --> int`  
    - 返回容器的长度（元素个数）  
    - eg: `len([1,2,3]) --> 3`  
  - `range(a,b) --> range对象(可迭代)`  
    - 生成从 a 到 b-1 的整数序列（惰性）  
    - eg: `list(range(1,4)) --> [1,2,3]`  
  - `zip(a,b) --> zip对象(可迭代的配对)`  
    - 将多个可迭代对象按元素配对  
    - eg: `list(zip([1,2],[3,4])) --> [(1,3),(2,4)]`  
  - `print(x) --> None`  
    - 将内容打印到控制台，副作用  
    - eg: `print('hello')` 输出 hello  
  - `round(x, 位数) --> float`  
    - 对浮点数四舍五入到指定小数位数  
    - eg: `round(3.14159,2) --> 3.14`


## 1. 公共实验组件

这一节只定义公共机制。首次打开或修改组件后，请执行 **Runtime → Restart session and run all**，避免 notebook 的旧状态污染结果。

### 1.1 环境、结果对象与随机数


In [ ]:
from dataclasses import dataclass
from math import erfc, exp, sqrt
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

# 中文字体补丁：本地与 Colab 都使用 Python 标准库下载，不依赖 wget
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.font_manager as fm

_FONT_PATH = Path("NotoSansCJKtc-Regular.otf")
_FONT_URL = (
    "https://raw.githubusercontent.com/notofonts/noto-cjk/main/"
    "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf"
)
if not _FONT_PATH.exists():
    try:
        urlretrieve(_FONT_URL, _FONT_PATH)
    except OSError as error:
        print(f"中文字体下载失败：{error}")
if _FONT_PATH.exists():
    fm.fontManager.addfont(_FONT_PATH)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False


@dataclass
class DynamicsResult:

    final_state: np.ndarray #(N,) -- 迭代结束时的状态
    sweeps: int             # 用了几轮/几步收敛（或到达上限）
    status: str             # 收敛方式，"fixed"/"cycle2"/"max_sweeps"/"max_steps"，
    flips_per_sweep: tuple[int, ...]# 每轮翻转次数，用于观察收敛速度，不参与后续计算
    state_history: Optional[list[np.ndarray]] = None #仅在显式要求时才非空，默认不保存，避免占内存
    energy_history: Optional[list[float]] = None


SEED = 0
# 全局默认种子，顶层实验没显式传别的种子时兜底


### 1.2 a：记忆生成

- `a1`：相互独立的随机记忆。
- `a2`：构造一对具有指定汉明距离的相似记忆。
- `a3`：围绕共同原型生成相关记忆。


In [ ]:
def a1_make_independent_memories(
    n: int,
    N: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 n -- 要生成的记忆条数；N -- 每条记忆的维度
    输出 memories:(n,N) -- n 条相互独立的随机 ±1 记忆
    """
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=(n, N))
    # [输入] 从 {-1,+1} 里独立采样，是后续所有存储/召回实验的起点


def a2_make_memories_with_close_pair(
    n: int,
    N: int,
    distance: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 n,N 同上；distance -- 最后一条记忆要跟第一条相差的比特数
    输出 memories:(n,N) -- 前 n-1 条互相独立，最后一条是第一条的近邻
    """
    if n < 2:
        raise ValueError("n 至少为 2")
        # [约束] 构造"一对相似记忆"至少要有两条记忆可比较

    memories = a1_make_independent_memories(n=n, N=N, rng=rng)
    # [中介变量] 先按完全随机生成，最后一条马上会被覆盖，只是占位

    memories[-1] = memories[0].copy()
    #［构造］在一堆随机记忆里，精确制造一对 特别相似的记忆，这里选第一条和最后一条，先拷贝

    flip_indices = rng.choice(N, size=distance, replace=False)
    # [中介变量] flip_indices:(distance,) -- 要翻转的比特位置，无放回抽样保证不重复翻同一位

    memories[-1, flip_indices] *= -1
    # [更新] 只翻这 distance 个位置，翻转后与 memories[0] 的汉明距离恰好等于 distance
    return memories


def a3_make_prototype_correlated_memories(
    n: int,
    N: int,
    flip_probability: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """
    输入 n,N 同上；flip_probability -- 每条记忆相对原型独立翻转每一位的概率
    输出 memories:(n,N) -- n 条共享同一原型、带统计相关性的记忆
         prototype:(N,) -- 共同原型，本身不是任何一条实际记忆
    """
    prototype = a1_make_independent_memories(n=1, N=N, rng=rng)[0]
    # [中介变量] 原型:(N,) -- 随机基准点

    memories = np.repeat(prototype[None, :], n, axis=0)
    # 把原型复制 n 份

    flip_mask = rng.random((n, N)) < flip_probability
    # [中介变量] flip_mask:(n,N) 布尔矩阵 -- 每个位置独立决定是否翻转：

    memories[flip_mask] *= -1
    # [更新] 按布尔矩阵 原地翻转对应位置
    return memories, prototype


### 1.3 b：权重构造与变换

`b1` 是标准 Hebb 权重；`b3`–`b6` 都是显式变换，因此实验代码能直接显示在标准权重上增加了什么。


In [ ]:
def b1_make_hebbian_weights(memories: np.ndarray) -> np.ndarray:
    """
    输入 memories:(n,N) -- n 条已生成好的记忆
    输出 weights:(N,N) -- 标准 Hebb 权重矩阵
    """
    weights = memories.astype(float).T @ memories.astype(float)
    # [存储] memories.T:(N,n) @ memories:(n,N) --> weights:(N,N)
    # [中介变量] astype(float)：来源 memories(int8)，用途防溢出(n=500 时求和会溢出)

    np.fill_diagonal(weights, 0.0)
    # [约束] T_ii=0。无自反馈，
    return weights


def b2_make_random_asymmetric_weights(
    N: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 N -- 神经元数
    输出 weights:(N,N) -- 无监督生成的随机权重，Tij 与 Tji 无关，用来测试对称性假设是否必要
    """
    weights = rng.uniform(-1.0, 1.0, size=(N, N))
    # [输入] 均匀采样，跟任何记忆无关
    np.fill_diagonal(weights, 0.0)
    # [约束] T_ii=0
    return weights


def b3_binarize_weights(weights: np.ndarray) -> np.ndarray:
    """
    输入 weights:(N,N) -- 任意已算好的权重（ b1 的输出）
    输出 binarized:(N,N) -- 每个非零元素被压成 ±1，只保留符号
    """
    binarized = np.sign(weights)
    # [更新] weights:(N,N) --> clipped:(N,N)，逐元素去数值取符号±1，这是式(3)"截断"的定义
    np.fill_diagonal(binarized, 0.0)
    # [约束] 显式再清一次对角线，保证截断后 T_ii=0 依然成立
    return binarized


def b4_keep_one_direction_per_pair(
    weights: np.ndarray,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """
    输入 weights:(N,N) -- 对称 Hebb 权重
    输出 one_way:(N,N) -- 每对 (i,j) 只保留一个方向的连接
         direction_mask:(N,N) -- True 表示对应的 weights[i,j] 被保留，即 j→i
    """
    one_way = np.zeros_like(weights)
    direction_mask = np.zeros(weights.shape, dtype=bool)
    # [中介变量] 权重矩阵和方向掩码先全零，随后逐对填入

    N = weights.shape[0]
    for i in range(N):
        for j in range(i + 1, N):
            # [实验控制] 每个无序神经元对只抽取一次方向
            if rng.random() < 0.5:
                one_way[i, j] = weights[i, j]
                direction_mask[i, j] = True
                # [更新] 保留 weights[i,j]，按局部场定义表示 j→i
            else:
                one_way[j, i] = weights[j, i]
                direction_mask[j, i] = True
                # [更新] 保留 weights[j,i]，按局部场定义表示 i→j
    return one_way, direction_mask


def b5_add_memory_with_saturation(
    weights: np.ndarray,
    memory: np.ndarray,
    limit: int = 3,
) -> np.ndarray:
    """
    输入 weights:(N,N) -- 当前累积权重；memory:(N,) -- 新来的一条记忆；limit -- 权重绝对值上限
    输出 updated:(N,N) -- 叠加新记忆并截幅后的权重
    """
    updated = weights + np.outer(memory, memory)
    # [更新] memory:(N,) 外积 memory:(N,) --> (N,N)，逐步累加，等价于持续学习
    np.fill_diagonal(updated, 0)
    # [约束] T_ii=0
    return np.clip(updated, -limit, limit)
    # [约束] 权重饱和在 [-limit, limit]。这是"自然遗忘"机制本身：
    # 旧记忆的贡献会被后续更新的 +1/-1 增量逐渐挤出这个有限区间


def b6_add_asymmetric_sequence_terms(
    weights: np.ndarray,
    memories: np.ndarray,
    strength: float,
) -> np.ndarray:
    """
    输入 weights:(N,N) -- 已有的对称 Hebb 权重；memories:(n,N) -- 要按顺序串起来的记忆序列；
        strength -- 序列项的强度 A
    输出 sequenced:(N,N) -- 叠加了非对称"下一条记忆"提示项的权重
    """
    sequenced = weights.astype(float).copy()
    # [中介变量] 复制而非原地改，避免污染调用者传入的 weights
    for current, following in zip(memories[:-1], memories[1:]):
        # [实验控制] 依次取相邻的 (V^s, V^{s+1}) 记忆对
        sequenced += strength * np.outer(following, current)
        # [更新] 式(13) ΔTij = A(2V^{s+1}_i-1)(2V^s_j-1)
        # following:(N,) 外积 current:(N,) --> (N,N)，following 在前、current 在后，
        # 这样局部场里 j 是"当前记忆"时才会推 i 往"下一条记忆"走，方向不能写反
    np.fill_diagonal(sequenced, 0.0)
    # [约束] T_ii=0
    return sequenced


### 1.4 c：初始状态


In [ ]:
def c1_start_from_memory(
    memories: np.ndarray,
    index: int,
) -> np.ndarray:
    """
    输入 memories:(n,N)；index -- 选第几条记忆当初始状态
    输出 state:(N,) -- 该记忆的副本
    """
    return memories[index].copy()
    # [输入] 复制，不是引用——避免后续 d1 原地修改污染 memories 数组


def c2_random_state(
    N: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 N -- 维度
    输出 state:(N,) -- 跟任何记忆都无关的随机初态
    """
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=N)
    # [输入] 用来测试"网络会漂到哪个吸引子"，跟 a1 生成记忆用的是同一种采样

# 扰动
def c3_perturb_memory(
    memory: np.ndarray,
    n_flips: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 memory:(N,) -- 一条干净记忆；n_flips -- 要翻转的比特数
    输出 state:(N,) -- 与 memory 相距恰好 n_flips 比特的初态
    """
    state = memory.copy()
    indices = rng.choice(memory.size, size=n_flips, replace=False)
    # [中介变量] indices:(n_flips,) -- 无放回抽样，保证真的翻了 n_flips 个不同位置
    state[indices] *= -1
    # [更新] 精确控制初始汉明距离，这是实验4"吸引盆半径"的自变量
    return state


def c4_unfamiliar_state(
    N: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 N；输出 state:(N,) -- "陌生输入"
    """
    return c2_random_state(N=N, rng=rng)
    # 这一步在问：c4 跟 c2 代码完全一样，为什么还要单独起个名字？
    # 答案：生成方式相同，但实验语境不同——c2 是"随便一个初态"，
    # c4 是"故意构造一个跟任何记忆都不像的输入"，命名区分的是实验意图，不是算法


### 1.5 d：动力学

`d1_run_async` 默认只返回终态和轻量摘要。只有明确设置 `record_states=True` 或 `record_energy=True` 时才保存过程数据。

注意：能量单调性只适用于对称权重。非对称实验不得把 `record_energy=True` 当作收敛证明。


In [ ]:
def _measure_energy(weights: np.ndarray, state: np.ndarray) -> float:
    """
    输入 weights:(N,N)；state:(N,)
    输出 标量 -- 式(7)定义的能量，仅在对称权重下单调不增，是观测量不是判据
    """
    return float(-0.5 * state @ weights @ state)
    # [观测·数据] 能量值；state:(N,) @ weights:(N,N) @ state:(N,) --> 标量
    # d1/d2 从不读这个函数的返回值来决定是否继续迭代，纯粹是外部拿来验证收敛性质用的尺子


def _resolve_threshold(threshold: float | np.ndarray, i: int) -> float:
    """
    输入 threshold -- 标量或长度 N 的数组；i -- 神经元编号
    输出 标量 -- 神经元 i 应用的阈值
    """
    return float(threshold if np.isscalar(threshold) else threshold[i])
    # [中介变量] 统一接口：调用方不用关心 threshold 到底是全局统一还是逐神经元


def d0_make_update_schedule(
    N: int,
    max_sweeps: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    输入 N -- 神经元数；max_sweeps -- 最大更新轮数
    输出 schedule:(max_sweeps,N) -- 每轮各自随机排列的神经元编号
    """
    return np.stack([rng.permutation(N) for _ in range(max_sweeps)])
    # [中介变量] 配对条件共享同一个 schedule；下一次试验仍重新随机生成


def d1_run_async(
    weights: np.ndarray,
    initial_state: np.ndarray,
    rng: np.random.Generator,
    update_schedule: Optional[np.ndarray] = None,
    max_sweeps: int = 50,
    threshold: float | np.ndarray = 0.0,
    record_states: bool = False,
    record_energy: bool = False,
) -> DynamicsResult:
    """
    输入 weights:(N,N) -- 已存好记忆的突触矩阵
        initial_state:(N,) -- 当前观察到的、可能带噪声的状态
        update_schedule:(max_sweeps,N) 或 None -- 可复用的随机更新日程
    输出 final_state:(N,) -- 迭代到不动点(或到达 max_sweeps)后的状态
    异步：每个神经元的更新立刻生效，同一轮内后面的神经元能看到前面的更新
    """
    if update_schedule is not None:
        update_schedule = np.asarray(update_schedule, dtype=int)
        expected_shape = (max_sweeps, initial_state.size)
        if update_schedule.shape != expected_shape:
            raise ValueError(f"update_schedule 形状应为 {expected_shape}，实际为 {update_schedule.shape}")
        # [约束] 每轮必须给出 N 个更新位置，轮数必须覆盖 max_sweeps

    state = initial_state.copy()
    # [中介变量] initial_state:(N,) --> state:(N,)，复制是为了不污染调用者传入的数组
    state_history = [state.copy()] if record_states else None
    # [观测·数据] 状态轨迹；默认 None，只有显式要求才保存完整轨迹，避免长时间实验内存爆掉
    energy_history = [_measure_energy(weights, state)] if record_energy else None
    # [观测·数据] 能量轨迹；同上，且只在对称权重下这条曲线才保证单调
    flips_per_sweep: list[int] = []
    # [中介变量] 每轮翻转次数的累计列表，用于观察收敛速度

    for sweep in range(1, max_sweeps + 1):
        flips = 0
        # [中介变量] 本轮计数器，只用于判停

        if update_schedule is None:
            order = rng.permutation(state.size)
            # [中介变量] 普通实验每轮重新随机排列，避免神经元编号偏置
        else:
            order = update_schedule[sweep - 1]
            # [实验控制] 配对条件读取同一轮日程，只消除更新顺序这一随机差异

        for i in order:
            # [中介变量] i:标量，本轮当前要更新的神经元编号

            local_field = weights[i] @ state - _resolve_threshold(threshold, i)
            # [更新] 式(2)左边；weights[i]:(N,); state:(N,) --> local_field:标量
            # 这一步在问：神经元 i 现在翻转，会不会更符合它记得的模式？
            # local_field 就是"其余神经元投票 i 该是+1还是-1"的净票数

            new_value = 1 if local_field > 0 else -1 if local_field < 0 else state[i]
            # [判断] local_field:标量 --> new_value:标量
            # 全文唯一非线性：票数过半就翻，没过半不翻，恰好为0时保持原值(阈值处不定义的边界约定)

            if new_value != state[i]:
                state[i] = new_value
                # [存储] 原地覆盖，立刻生效——这是异步和 d2 同步版本的分界线
                flips += 1
                # [中介变量] 计数器递增
                if record_states:
                    state_history.append(state.copy())
                    # [观测·数据] 状态快照；只在显式要求时才追加，逐次翻转记录，比逐轮记录更细
                if record_energy:
                    energy_history.append(_measure_energy(weights, state))
                    # [观测·数据] 能量快照；同上

        flips_per_sweep.append(flips)
        # [中介变量] 记录本轮总翻转数

        if flips == 0:
            # [判断] 一整轮没有任何神经元翻转 = 到达不动点
            return DynamicsResult(
                final_state=state,
                sweeps=sweep,
                status="fixed",
                flips_per_sweep=tuple(flips_per_sweep),
                state_history=state_history,
                energy_history=energy_history,
            )

    return DynamicsResult(
        final_state=state,
        sweeps=max_sweeps,
        status="max_sweeps",
        # [判断] 到达轮数上限仍未收敛，可能是周期或混沌游走(见实验1)
        flips_per_sweep=tuple(flips_per_sweep),
        state_history=state_history,
        energy_history=energy_history,
    )


def d2_run_sync(
    weights: np.ndarray,
    initial_state: np.ndarray,
    max_steps: int = 100,
    record_states: bool = False,
) -> DynamicsResult:
    """
    输入 weights:(N,N)；initial_state:(N,)
    输出 final_state:(N,) -- 同步更新版本，作为论文外机制对照，不是论文本身用的算法
    同步：本轮所有神经元一起根据"上一轮"的状态决定新值，因此可能出现二周期
    """
    state = initial_state.copy()
    previous = None
    # [中介变量] 保存上一轮状态，用于检测二周期(A→B→A→B)
    state_history = [state.copy()] if record_states else None
    flips_per_step: list[int] = []

    for step in range(1, max_steps + 1):
        local_fields = weights @ state
        # [更新] weights:(N,N) @ state:(N,) --> local_fields:(N,)
        # 一次性对所有神经元算局部场，用的全是上一轮的 state，
        # 这跟 d1 逐个用"当前最新 state"算，是同步和异步的根本区别

        new_state = state.copy()
        new_state[local_fields > 0] = 1
        new_state[local_fields < 0] = -1
        # [判断] local_fields:(N,) --> new_state:(N,)，逐元素判决，一次性全部生效

        flips_per_step.append(int(np.count_nonzero(new_state != state)))
        # [中介变量] 统计本步翻转数，仅观测用
        if record_states:
            state_history.append(new_state.copy())
            # [观测·数据] 状态快照

        if np.array_equal(new_state, state):
            # [判断] 与上一状态相同 = 不动点
            return DynamicsResult(
                new_state, step, "fixed", tuple(flips_per_step), state_history
            )
        if previous is not None and np.array_equal(new_state, previous):
            # [判断] 与上上一状态相同 = 二周期，同步更新特有的失败模式，异步版本不会出现
            return DynamicsResult(
                new_state, step, "cycle2", tuple(flips_per_step), state_history
            )
        previous = state.copy()
        # [中介变量] 滚动保存"上一轮"状态供下一次判周期用
        state = new_state

    return DynamicsResult(
        state, max_steps, "max_steps", tuple(flips_per_step), state_history
    )


### 1.6 e：测量

过程型指标尽量在线累计，不默认保存完整轨迹。`e4_initial_flip_count` 只执行指定次数的神经元更新尝试，返回实际发生的状态调整次数，对应论文定义的初始处理速率；`e6_signal_noise_samples` 返回总体统计所需的逐神经元信号和噪声样本。

In [ ]:
def e1_hamming_distance(
    state_a: np.ndarray,
    state_b: np.ndarray,
) -> int:
    """
    输入 state_a,state_b:(N,)
    输出 标量 -- 不同比特位的个数
    """
    return int(np.count_nonzero(state_a != state_b))
    # [观测·数据] 汉明距离；逐元素比较后计数，是最基础的召回误差量尺


def e2_identify_attractor(
    state: np.ndarray,
    memories: np.ndarray,
) -> tuple[int, int]:
    """
    输入 state:(N,) -- 网络当前状态；memories:(n,N) -- 候选记忆集合(可以是名义记忆也可以是±memories)
    输出 (index, distance) -- 离 state 最近的那条记忆的编号和距离
    """
    distances = np.count_nonzero(memories != state, axis=1)
    # [观测·数据] 逐条记忆的汉明距离；memories:(n,N) != state:(N,) 广播比较 --> (n,N) 布尔 --> axis=1 求和 --> (n,)
    index = int(np.argmin(distances))
    # [判断] 取距离最小的那条
    return index, int(distances[index])


def e3_effective_state_count(
    state_history: list[np.ndarray],
) -> float:
    """
    输入 state_history -- d1 record_states=True 时保存的轨迹
    输出 标量 M -- 式(9)有效状态数 exp(熵)，用来量化"游走覆盖了多大范围"
    """
    if not state_history:
        return 0.0
    counts: dict[bytes, int] = {}
    # [中介变量] 用状态的字节表示做字典键，统计每个不同状态出现了几次
    for state in state_history:
        key = state.astype(np.int8, copy=False).tobytes()
        # [中介变量] ndarray 本身不能当字典键(不可哈希)，转成 bytes 才能计数
        counts[key] = counts.get(key, 0) + 1

    probabilities = np.array(list(counts.values()), dtype=float)
    probabilities /= probabilities.sum()
    # [中介变量] 频次归一化成经验概率分布 p_i

    entropy = -float(np.sum(probabilities * np.log(probabilities)))
    # [观测·结果] 熵；式(9) ln M = -Σ p_i ln p_i 的右边先算出熵
    return exp(entropy)
    # [观测·结果] 有效状态数 M；熵取指数还原成"等效状态数" M


def e4_initial_flip_count(
    weights: np.ndarray,
    initial_state: np.ndarray,
    rng: np.random.Generator,
    attempts: int,
    threshold: float = 0.0,
    update_order: Optional[np.ndarray] = None,
) -> int:
    """
    输入 weights,initial_state 同 d1；attempts -- 短时间窗内的更新尝试次数
    输出 整数 -- 这些尝试中实际发生的神经元状态调整次数
    """
    state = initial_state.copy()
    flips = 0
    if update_order is None:
        order = rng.integers(0, state.size, size=attempts)
        # [随机异步] 有放回抽样；同一神经元在短时间窗内可能被再次选中
    else:
        order = np.asarray(update_order, dtype=int)[:attempts]
        if order.size != attempts:
            raise ValueError("update_order 的长度不能小于 attempts")
        # [实验控制] 配对输入共享同一更新事件序列，排除更新时机差异

    for i in order:
        local_field = weights[i] @ state - threshold
        # [更新] 与 d1 相同：当前位置根据当前最新状态计算局部场
        new_value = 1 if local_field > 0 else -1 if local_field < 0 else state[i]
        if new_value != state[i]:
            state[i] = new_value
            # [存储] 异步更新立即生效
            flips += 1
            # [观测·结果] 固定时间窗内实际发生的状态调整次数
    return flips


def e5_nearest_memory_path(
    state_history: list[np.ndarray],
    memories: np.ndarray,
) -> list[int]:
    """
    输入 state_history -- 轨迹；memories:(n,N)
    输出 path -- 依次经过的、且与上一个不同的最近记忆编号序列
    """
    path: list[int] = []
    for state in state_history:
        index, _ = e2_identify_attractor(state, memories)
        # [观测·数据] 当前最近记忆编号；每一帧都判一次"当前离哪条记忆最近"
        if not path or path[-1] != index:
            path.append(index)
            # [中介变量] 只在编号发生变化时才追加，把长轨迹压缩成"经过了哪些记忆区域"
    return path


def e6_signal_noise_samples(
    weights: np.ndarray,
    target: np.ndarray,
    signal_counts: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    输入 weights:(N,N)；target:(N,)；signal_counts:(N,) -- 各神经元收到的目标记忆信号项数
    输出 signal,noise:(N,) -- 供多次试验汇总后计算总体 SNR 的逐神经元样本
    """
    weights_float = np.asarray(weights, dtype=np.float64)
    target_float = np.asarray(target, dtype=np.float64)
    signal = np.asarray(signal_counts, dtype=np.float64)
    # [类型转换] 状态仍保存为 int8；只在连续测量边界转成 float64，避免大整数乘法溢出

    if signal.shape != target_float.shape:
        raise ValueError("signal_counts 必须与 target 形状相同")
    # [约束] 每个神经元必须对应一个信号项数

    aligned_total_input = target_float * (weights_float @ target_float)
    # [观测·数据] 乘回 target 符号后，正确记忆方向上的输入统一为正
    noise = aligned_total_input - signal
    # [观测·数据] 总输入减去目标记忆本身的精确信号，剩余为其他记忆的交叉干扰
    return signal, noise


def theoretical_bit_error_probability(n: int, N: int) -> float:
    """
    输入 n -- 存储记忆数；N -- 维度
    输出 标量 -- 式(10)单比特出错概率的高斯近似
    """
    if n <= 1:
        return 0.0
        # [约束] 只存 1 条记忆时没有交叉干扰项，误差恒为0
    sigma = sqrt((n - 1) * N / 2)
    # [中介变量] 式(10)里噪声项的标准差 σ = [(n-1)N/2]^(1/2)
    return 0.5 * erfc((N / 2) / (sigma * sqrt(2)))
    # [观测·结果] 理论单比特出错概率；高斯尾部概率，用互补误差函数算，是理论曲线，不是模拟结果

### 1.7 f：展示


In [ ]:
def f1_plot_curve(
    x,
    series: dict[str, list[float] | np.ndarray],
    xlabel: str,
    ylabel: str,
    title: str,
):
    """
    输入 x -- 横轴取值；series -- {曲线名: 纵轴值}，可以同时画多条线
    输出 无返回值，直接 plt.show()
    """
    fig, ax = plt.subplots(figsize=(7, 4))
    for label, values in series.items():
        ax.plot(x, values, marker="o", label=label)
        # [展示] 纯画图，不影响任何实验数值
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    if len(series) > 1:
        ax.legend()
    plt.show()


def f2_plot_bar(
    labels,
    values,
    ylabel: str,
    title: str,
):
    """
    输入 labels -- 类别名；values -- 对应柱高
    输出 无返回值
    """
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    # [展示]
    ax.set(ylabel=ylabel, title=title)
    plt.xticks(rotation=0, ha="center")
    plt.show()


## 2. 公共组件自检

这些检查在跑论文实验前先验证：编码只能是 $\{-1,+1\}$、Hebb 权重对称且对角线为零、异步更新时对称系统的能量不增加、轨迹默认不保存、同步更新能识别二周期。


In [ ]:
rng = np.random.default_rng(SEED)
memories = a1_make_independent_memories(n=5, N=30, rng=rng)
weights = b1_make_hebbian_weights(memories)
initial_state = c3_perturb_memory(memories[0], n_flips=4, rng=rng)
run = d1_run_async(
    weights,
    initial_state,
    rng,
    record_energy=True,
)

assert set(np.unique(memories)) <= {-1, 1}
# [约束检验] 编码只能是 {-1,+1}
assert set(np.unique(run.final_state)) <= {-1, 1}
# [约束检验] 动力学不会产生编码之外的值
assert np.allclose(weights, weights.T)
# [约束检验] Hebb 权重对称
assert np.allclose(np.diag(weights), 0)
# [约束检验] 对角线为零
assert run.state_history is None
# [约束检验] 默认不保存轨迹(这次调用没传 record_states=True)
assert np.all(np.diff(run.energy_history) <= 1e-9)
# [约束检验] 对称权重下能量单调不增，允许 1e-9 的浮点误差

one_way, direction_mask = b4_keep_one_direction_per_pair(weights, rng)
assert np.count_nonzero(direction_mask) == weights.shape[0] * (weights.shape[0] - 1) // 2
# [约束检验] 每个无序神经元对恰好保留一个连接方向
assert np.all(one_way[~direction_mask] == 0)
# [约束检验] 方向掩码之外的单向权重全部为零

shared_schedule = d0_make_update_schedule(N=30, max_sweeps=50, rng=rng)
shared_run_a = d1_run_async(weights, initial_state, rng, update_schedule=shared_schedule)
shared_run_b = d1_run_async(weights, initial_state, rng, update_schedule=shared_schedule)
assert np.array_equal(shared_run_a.final_state, shared_run_b.final_state)
# [约束检验] 相同权重、初态和更新日程必须产生相同轨迹终点

cycle_weights = np.array([[0.0, 1.0], [1.0, 0.0]])
# [输入] 手工构造一个最小反例：两个神经元互相正向连接
cycle_run = d2_run_sync(
    cycle_weights,
    np.array([1, -1], dtype=np.int8),
)
assert cycle_run.status == "cycle2"
# [约束检验] 同步更新下这个反例必然产生二周期，验证 d2 能正确识别它

print("公共组件自检通过")


## 实验1：非对称随机连接的动力学行为


**问题**：移除 $T_{ij}=T_{ji}$ 后，随机初态会收敛、进入周期，还是在局部区域游走？

**配方**：`b2 → c2 → d1 → e3 → f2`

论文使用 $N=30$ 和 $N=100$。下面先重复 50 个随机初态，再单独记录一次过程轨迹；密集重复部分不保存轨迹。论文这一段使用 $\{0,1\}$ 状态与零阈值；转换到代码的 $\{-1,+1\}$ 状态后，等价阈值必须变成每行权重和的相反数，而不能继续直接写成零。


In [ ]:
N = 30
weight_seeds = [2, 7, 13, 21, 34]
# *[输入] 本实验的自变量：5个独立的随机非对称权重矩阵种子。
# 上一版只用了单个种子(2)，跑出0条"简单循环"，无法判断是这个种子运气不好，
# 还是简单循环本来就稀有——单种子的计数结论撑不住这个问题，需要多种子汇总。


# 1.运行轨迹----------
all_statuses = []
all_non_fixed_histories = []
per_seed_fixed_count = {}
# [中介变量] 每个种子各自的固定点占比，用来看不同种子之间是否稳定

for seed in weight_seeds: # [外层]：5个种子，5 种随机非对称权重矩阵
    seed_rng = np.random.default_rng(seed)
    weights = b2_make_random_asymmetric_weights(N=N, rng=seed_rng)

    threshold = -weights.sum(axis=1)
    # [编码翻译] 论文原文用 {0,1} 编码、U=0；这里代码统一用 {-1,+1}，

    fixed_this_seed = 0
    for _ in range(50): # [内层] 50 次新的随机初态
        initial_state = c2_random_state(N=N, rng=seed_rng)
        run = d1_run_async(
            weights=weights,
            initial_state=initial_state,
            rng=seed_rng,
            max_sweeps=50,
            threshold=threshold,
            record_states=True,
            # [实验控制] 每条轨迹都记录，供后面统一算 M/χ
        )
        all_statuses.append(run.status)
        if run.status == "fixed":
            fixed_this_seed += 1
        else:
            all_non_fixed_histories.append(run.state_history)
            # [中介变量] 只有非固定点轨迹才需要留着算 M/χ

    per_seed_fixed_count[seed] = fixed_this_seed

# 2.汇总状态----------
total_trials = len(all_statuses)
total_fixed = all_statuses.count("fixed")
# [观测·结果] 跨种子汇总固定点数量，最终统一显示

# 3.计算M与χ----------
M_values = []
chi_values = []
for history in all_non_fixed_histories:
    M = e3_effective_state_count(history)
    # [观测·数据] 有效状态数 M(式9)，熵加权的"访问过多少不同状态"

    total_steps = len(history)
    unique_states = len(set(state.astype(np.int8).tobytes() for state in history))
    chi = 1.0 - (unique_states / total_steps)
    # [观测·数据] 混沌系数 χ：1 - 去重状态数/总步数，越接近0越混沌

    M_values.append(M)
    chi_values.append(chi)

CYCLE_LIKE_M_THRESHOLD = 10
# [约束·可执行定义] 只当参考线画出来，不再用它把结果强行分进两个桶——
# 单一阈值在单一种子上"卡出0个"的问题，根源就是把连续分布硬切成了离散计数

n_below_threshold = sum(1 for m in M_values if m <= CYCLE_LIKE_M_THRESHOLD)
# [观测·结果] 汇总多种子后的疑似循环计数，最终统一显示

# 4.绘图----------
fig, axes = plt.subplots(2, 1, figsize=(7, 6))

axes[0].hist(M_values, bins=30)
axes[0].set_xlabel("M (有效状态数)")
axes[0].set_ylabel("轨迹数")
axes[0].set_title(
    f"非固定点轨迹的 M 分布（{len(weight_seeds)}个种子，共{len(M_values)}条）"
)
# [展示]

axes[1].hist(chi_values, bins=30)
axes[1].set_xlabel("χ (混沌系数)")
axes[1].set_ylabel("轨迹数")
axes[1].set_title("同一批轨迹的 χ 分布")
# [展示]

plt.tight_layout()
plt.show()

# 5.结果数据----------
print("各权重种子的固定点次数：", per_seed_fixed_count)
print(f"合计 {total_trials} 次：固定点 {total_fixed} 条，未固定 {total_trials - total_fixed} 条")
print(f"未固定轨迹的 M：均值 {np.mean(M_values):.2f}，中位数 {np.median(M_values):.2f}")
print(f"未固定轨迹的 χ：均值 {np.mean(chi_values):.3f}，中位数 {np.median(chi_values):.3f}")
print(f"M≤{CYCLE_LIKE_M_THRESHOLD} 的疑似简单循环：{n_below_threshold} 条")


**对照论文**：最常见行为是稳定点，偶见二周期、局部混沌游走；$N=30$ 的一次局部游走得到有效状态数 $M=25$。

**结果分析**：非对称权重下，固定点与持续游走同时出现；本次未固定轨迹中没有明显的短简单循环。


## 实验2：存储容量与信噪比

**原文定位**：论文 pp.2556–2557，图2；原文从 “Computer modeling” 开始描述容量模拟。

**问题**：当 $N=100$ 时，随着存储记忆数 $n$ 增加，正确召回如何下降？

**配方**：`a1 → b1 → c1 → d1 → e1 → f1`

每次试验都重新生成记忆和权重，运行后只保留终态误差；不会累计状态轨迹。


In [ ]:
# 1.容量扫描----------
rng = np.random.default_rng(102)
N = 100
n_values = np.arange(1, 21)
trials_per_n = 50

mean_errors = []
exact_rates = []
theory_exact_rates = []
errors_by_n = {}

for n in n_values:
    errors = []
    for _ in range(trials_per_n):
        memories = a1_make_independent_memories(n=n, N=N, rng=rng)
        weights = b1_make_hebbian_weights(memories)

        target_index = int(rng.integers(n))
        initial_state = c1_start_from_memory(memories, target_index)

        run = d1_run_async(weights, initial_state, rng)
        errors.append(e1_hamming_distance(run.final_state, memories[target_index]))

    errors = np.asarray(errors)
    if n in (5, 10, 15):
        errors_by_n[int(n)] = errors.copy()
    mean_errors.append(float(errors.mean()))
    exact_rates.append(float(np.mean(errors == 0)))

    bit_error = theoretical_bit_error_probability(n=n, N=N)
    theory_exact_rates.append((1 - bit_error) ** N)

# 2.论文直方图----------
import matplotlib.pyplot as plt
import numpy as np

BINS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 101]
BIN_LABELS = ['', '', '', '3', '', '', '6', '', '', '9', '10-19', '20-29', '30-39', '40-49', '>49']

def compute_histogram(errors):
    hist, _ = np.histogram(errors, bins=BINS)
    return hist / len(errors)

fig, axes = plt.subplots(len(errors_by_n), 1, figsize=(9, 8), sharex=True)

for ax, n in zip(axes, sorted(errors_by_n)):
    probabilities = compute_histogram(errors_by_n[n])
    ax.bar(range(len(BIN_LABELS)), probabilities, width=0.8)
    ax.text(0.22, 0.64, f"n = {n}\nN = {N}", transform=ax.transAxes)
    # [展示] 按论文版式把 n 与 N 写在各面板内部
    ax.set_ylim(0, 1.05)

axes[-1].set_xticks(range(len(BIN_LABELS)))
axes[-1].set_xticklabels(BIN_LABELS)
axes[-1].set_xlabel('错误位数')
fig.supylabel('概率')

plt.tight_layout()
plt.show()

# 3.结果数据----------
for n in (5, 10, 15):
    print(
        f"n={n}：精确召回率={exact_rates[n - 1]:.3f}，"
        f"平均错误位数={mean_errors[n - 1]:.2f}"
    )

**对照论文**：论文报告 $n=5$ 时记忆几乎总是稳定；在 $n=15\approx0.15N$ 时，约一半记忆仍接近原记忆，另一半严重失真。有限试验次数和随机种子会造成曲线波动。

**结果分析**：存储记忆增多时，精确召回率下降，错误位数分布向较大值移动。


## 实验3：随机初态的吸引子分布

**原文定位**：论文 p.2557，“Given some arbitrary starting state …” 段落。

**问题**：随机初态最终精确落入原始记忆、反相记忆，还是未存储的伪吸引子？

**配方**：`a1 → b1 → c2 → d1 → e2 → f2`


In [ ]:
memory_rng = np.random.default_rng(71)
trajectory_rng = np.random.default_rng(103)
N, n, trials = 30, 5, 200
memories = a1_make_independent_memories(n=n, N=N, rng=memory_rng)
weights = b1_make_hebbian_weights(memories)

nominal_attractors = np.vstack([memories, -memories])
# [中介变量] 前 n 条是原始记忆，后 n 条是全局反相记忆。
# 论文把两者合称为 10 个名义稳定状态，本实验额外拆开计数以检查对称性。

outcomes = {"原始记忆": 0, "反相记忆": 0, "伪吸引子": 0}

nominal_stability = []
for index, attractor in enumerate(nominal_attractors):
    check = d1_run_async(
        weights,
        attractor,
        np.random.default_rng(10_000 + index),
        max_sweeps=200,
    )
    nominal_stability.append(
        check.status == "fixed" and np.array_equal(check.final_state, attractor)
    )
if not all(nominal_stability):
    raise RuntimeError("当前记忆实例不具备 10 个名义稳定状态")
# [实验前提] 先验证 5 条记忆及其 5 条反相确实都是固定点。

for _ in range(trials):
    initial_state = c2_random_state(N=N, rng=trajectory_rng)
    # *[输入] 每次从独立的随机状态开始。

    run = d1_run_async(weights, initial_state, trajectory_rng, max_sweeps=200)
    if run.status != "fixed":
        raise RuntimeError("对称权重网络未在 200 轮内到达固定点")
    # [约束] 只对真正收敛的稳定状态进行吸引子分类。

    index, distance = e2_identify_attractor(run.final_state, nominal_attractors)
    # [观测·数据] e2 返回最近候选编号与整数汉明距离。

    if distance == 0 and index < n:
        outcomes["原始记忆"] += 1
    elif distance == 0:
        outcomes["反相记忆"] += 1
    else:
        outcomes["伪吸引子"] += 1
        # [判断] 即使只差 1~2 bit，只要不是精确命中，就是未存储的稳定点。

f2_plot_bar(
    list(outcomes),
    [100 * value / trials for value in outcomes.values()],
    ylabel="试验占比 (%)",
    title="随机初态收敛到的吸引子类型",
)

nominal_count = outcomes["原始记忆"] + outcomes["反相记忆"]
print(f"原始记忆：{outcomes['原始记忆']}/{trials}")
print(f"反相记忆：{outcomes['反相记忆']}/{trials}")
print(f"伪吸引子：{outcomes['伪吸引子']}/{trials}")
print(f"名义稳定状态合计：{nominal_count}/{trials}（{nominal_count / trials:.1%}）")


**对照论文**：论文在 $N=30,n=5$ 时将 5 条原始记忆及其 5 条全局反相统称为 10 个“名义稳定状态”；约 85% 的试验结束于这些状态，约 10% 结束于无明显意义的稳定状态，约 5% 结束于非常接近名义记忆的稳定状态。本图将后两者统一计为“伪吸引子”。

**结果分析**：只有汉明距离为 0 才是精确命中已存记忆；“原始记忆”和“反相记忆”的拆分用于展示全局反转对称性，与论文比较时应使用两者的合计占比。原记忆种子 103 只有 8/10 个名义状态真正稳定，因而精确命中率仅约 28.5%；该偏差不是由更新轮数不足造成。本次改用经 10/10 稳定性检查的固定记忆实例。


## 实验4：吸引盆与纠错能力

**原文定位**：论文 p.2557，“partially random starting states” 段落。

**问题**：从已存记忆翻转多少 bit 后，网络仍能回到最近的原记忆？

**配方**：`a1 → b1 → c3 → d1 → e2 → f1`


In [ ]:
rng = np.random.default_rng(104)
N, n = 30, 5
distances = np.arange(0, 13)
# *[输入] 本实验的自变量：初始扰动的汉明距离，从0扫到12
trials_per_distance = 100
memories = a1_make_independent_memories(n=n, N=N, rng=rng)
weights = b1_make_hebbian_weights(memories)
recall_rates = []

for distance in distances:
    successes = 0
    for _ in range(trials_per_distance):
        target_index = int(rng.integers(n))
        initial_state = c3_perturb_memory(
            memories[target_index],
            n_flips=int(distance),
            rng=rng,
        )
        # [输入] 精确控制起点离目标记忆有多远

        run = d1_run_async(weights, initial_state, rng)
        nearest_index, _ = e2_identify_attractor(run.final_state, memories)
        # [观测·数据] 终态最近的记忆编号；只关心落到哪条记忆最近，不关心具体距离

        successes += int(nearest_index == target_index)
        # [判断] 是否回到了"出发时那条"记忆，而不是随便一条记忆
    recall_rates.append(successes / trials_per_distance)

f1_plot_curve(
    distances,
    {"回到目标记忆的概率": recall_rates},
    xlabel="初始汉明距离",
    ylabel="召回概率",
    title="吸引盆大小与纠错能力",
)

print("初始汉明距离 → 召回概率")
print("，".join(f"{int(distance)}→{rate:.3f}" for distance, rate in zip(distances, recall_rates)))


**对照论文**：论文报告距离不超过 5 bit 时返回最近记忆的概率超过 90%，距离 12 时下降到约 0.2。随机记忆集合不同会明显改变单次曲线。

**结果分析**：吸引盆半径大约在 5 到 8 之间


## 实验5：截断权重

**原文定位**：论文 p.2557，“clipped $T_{ij}$” 段落。

**问题**：把 Hebb 权重压成 $\{-1,+1\}$ 后，容量和误差怎样变化？

**配方**：`a1 → b1+b3 → c1 → d1 → e1 → f2`


In [ ]:
rng = np.random.default_rng(105)
N, trials = 100, 100
conditions = [
# *[输入] 本实验的自变量：是否截断权重 + 负载 n 的组合，三种条件
    ("标准权重 n=12", 12, False),
    ("截断权重 n=9", 9, True),
    ("截断权重 n=13", 13, True),
]
# [输入] 三种条件：普通权重用较少记忆(12)，截断权重分别测低负载(9)和论文报告的信息量最优负载(13)
condition_errors = []

for _, n, clipped in conditions:
    errors = []
    for _ in range(trials):
        memories = a1_make_independent_memories(n=n, N=N, rng=rng)
        weights = b1_make_hebbian_weights(memories)
        if clipped:
            weights = b3_binarize_weights(weights)
            # [更新] 只在这一分支把权重压成 {-1,0,+1}，标准分支保持浮点权重不变

        target_index = int(rng.integers(n))
        initial_state = c1_start_from_memory(memories, target_index)
        run = d1_run_async(weights, initial_state, rng)
        errors.append(e1_hamming_distance(run.final_state, memories[target_index]))
    condition_errors.append(float(np.mean(errors)))

f2_plot_bar(
    [name for name, _, _ in conditions],
    condition_errors,
    ylabel="平均错误位数",
    title="标准权重与截断权重",
)

for (name, _, _), error in zip(conditions, condition_errors):
    print(f"{name}：平均错误位数={error:.2f}")

**对照论文**：论文报告 $N=100$ 时，截断权重的 $n=9$ 与普通权重的 $n=12$ 错误水平相近；截断权重的最大 Shannon 信息量约出现在 $n=13$。

**结果分析**：截断只保留权重符号；低负载下仍可工作，但负载升高时错误通常更明显。


## 机制扩展1：有限权重与自然遗忘

**原文定位**：论文 p.2557，“The saturation of the possible size” 段落。

**问题**：权重限制在 $0,\pm1,\pm2,\pm3$ 时，新记忆是否逐渐覆盖旧记忆？

**配方**：`a1 → b5 → c1 → d1 → e1 → f1`


In [ ]:
rng = np.random.default_rng(106)
N, stream_length = 100, 80
memory_stream = a1_make_independent_memories(n=stream_length, N=N, rng=rng)
# [输入] 一次性生成80条记忆，模拟"依次到来"的记忆流

weights = np.zeros((N, N), dtype=float)
# [中介变量] 权重从零开始累积，不是一次性 Hebb 算好的

for memory in memory_stream:
    weights = b5_add_memory_with_saturation(weights, memory, limit=3)
    # *[更新] 本实验的自变量：饱和截幅的权重累积；越晚加入的记忆影响越大

ages = np.arange(stream_length - 1, -1, -1)
# [中介变量] 把记忆流的位置转换成"年龄"：最后加入的记忆年龄为0(最新)，最早加入的年龄最大

errors_by_age = []
for memory in memory_stream:
    initial_state = c1_start_from_memory(memory[None, :], 0)
    # [编码翻译] c1 要求二维数组做索引，这里单条记忆临时升维成 (1,N) 再取第0条

    run = d1_run_async(weights, initial_state, rng)
    errors_by_age.append(e1_hamming_distance(run.final_state, memory))
    # [观测·数据] 单条记忆的召回误差；拿饱和后的最终权重，逐条测每条记忆现在还能不能被稳定召回

order = np.argsort(ages)
# [中介变量] 排序索引，因为 memory_stream 是按时间顺序而不是年龄顺序排的

f1_plot_curve(
    ages[order],
    {"召回误差": np.asarray(errors_by_age)[order]},
    xlabel="存活轮数，0=最新",
    ylabel="错误位数",
    title="有限权重下的自然遗忘",
)

print(f"最新记忆：错误位数={np.asarray(errors_by_age)[ages == 0].mean():.2f}")
print(f"最早记忆：错误位数={np.asarray(errors_by_age)[ages == ages.max()].mean():.2f}")


**解释**：这是论文给出的硬件机制演示，不是带完整统计表的独立实验。曲线应总体表现为旧记忆更容易失真，但单条随机记忆会有波动。

**结果分析**：权重饱和使后加入记忆不断覆盖早期贡献，较旧记忆通常更难召回。


## 实验6：单向连接与软失效

**原文定位**：论文 p.2557，“Simulations were carried out with only one ij connection” 段落。

**问题**：若每对神经元只保留一个连接方向，稳定记忆是否仍存在？

**配方**：`a1 → b1+b4 → c1 → d1 → e1+e6 → f2`


In [ ]:
# 1.实验设置----------
rng = np.random.default_rng(107)
N, n, trials = 100, 10, 100
# [输入·固定] N=100、n=10，每种连接重复100次

errors = {"对称连接": [], "单向连接": []}
symmetric_signal_samples = []
symmetric_noise_samples = []
one_way_signal_samples = []
one_way_noise_samples = []
# [中介变量] 分别累计召回误差，以及所有试验的逐神经元信号和噪声样本


# 2.配对试验----------
for _ in range(trials):
    memories = a1_make_independent_memories(n=n, N=N, rng=rng)
    symmetric = b1_make_hebbian_weights(memories)
    one_way, direction_mask = b4_keep_one_direction_per_pair(symmetric, rng)
    # *[更新] 本实验的自变量：是否把每个双向连接对改成只保留一个随机方向
    # *[实验控制] 单向权重直接由本次对称权重变换，连接方向是两种条件之间的唯一结构差异

    target_index = int(rng.integers(n))
    target = memories[target_index]
    initial_state = c1_start_from_memory(memories, target_index)
    # [输入] 两种连接使用同一条目标记忆和同一个初始状态

    update_schedule = d0_make_update_schedule(N=N, max_sweeps=50, rng=rng)
    # [实验控制] 每次试验重新随机生成日程，但本次的两种连接共享完全相同的更新顺序

    for label, weights in [
        ("对称连接", symmetric),
        ("单向连接", one_way),
    ]:
        run = d1_run_async(
            weights,
            initial_state,
            rng,
            update_schedule=update_schedule,
        )
        errors[label].append(e1_hamming_distance(run.final_state, target))
        # [观测·数据] 从完全相同的起点和更新日程出发，记录最终错误位数

    symmetric_signal_counts = np.full(N, N - 1, dtype=np.float64)
    one_way_signal_counts = direction_mask.sum(axis=1)
    # [中介变量] 对称连接每个神经元收到 N-1 项目标信号；单向连接按实际入度逐行计数

    symmetric_signal, symmetric_noise = e6_signal_noise_samples(
        symmetric,
        target,
        symmetric_signal_counts,
    )
    one_way_signal, one_way_noise = e6_signal_noise_samples(
        one_way,
        target,
        one_way_signal_counts,
    )
    symmetric_signal_samples.extend(symmetric_signal)
    symmetric_noise_samples.extend(symmetric_noise)
    one_way_signal_samples.extend(one_way_signal)
    one_way_noise_samples.extend(one_way_noise)
    # [观测·数据] e6 在测量边界转 float64；这里汇总100次试验的全部神经元样本


# 3.绘图----------
mean_errors = {
    name: float(np.mean(values))
    for name, values in errors.items()
}
f2_plot_bar(
    list(mean_errors),
    list(mean_errors.values()),
    ylabel="平均错误位数",
    title="单向连接下的软失效",
)


# 4.结果数据----------
symmetric_snr = np.mean(symmetric_signal_samples) / np.std(symmetric_noise_samples)
one_way_snr = np.mean(one_way_signal_samples) / np.std(one_way_noise_samples)
snr_ratio = float(one_way_snr / symmetric_snr)
# [观测·结果] 先汇总全部样本再算总体 SNR，对应论文的信号强度/噪声标准差

print(
    f"平均错误位数：对称连接={mean_errors['对称连接']:.2f}，"
    f"单向连接={mean_errors['单向连接']:.2f}"
)
print(f"SNR 比值（单向/对称）= {snr_ratio:.3f}")
print(f"理论值 1/sqrt(2) = {1 / np.sqrt(2):.3f}")


**对照论文**：论文观察到错误率提高，但算法仍产生稳定极小值；信噪比按约 $1/\sqrt{2}$ 降低，而不是突然完全失效。

**结果分析**：每对连接只保留一个方向后，召回误差增大；但网络并非立刻完全失效。


## 实验7：相似记忆的混淆与融合

**原文定位**：论文 p.2557，$N=100,n=8$，令第八条记忆与另一条的汉明距离为 30、20 或 10。

**问题**：两条记忆太相似时，它们保持分离、发生偏移，还是融合成同一吸引子？

**配方**：`a2 → b1 → c1 → d1 → e1/e2 → f1`


In [ ]:
rng = np.random.default_rng(108)
N, n, trials = 100, 8, 60
distances = [30, 20, 10]
# *[输入] 本实验的自变量：那对相似记忆之间的汉明距离，论文点名的三个档位
fused_rates = []
displaced_errors = []

for distance in distances:
    fused = 0
    total_error = 0
    for _ in range(trials):
        memories = a2_make_memories_with_close_pair(
            n=n,
            N=N,
            distance=distance,
            rng=rng,
        )
        # [存储] 前 n-1 条互相独立，最后一条精确地跟第一条相距 distance

        weights = b1_make_hebbian_weights(memories)
        update_schedule = d0_make_update_schedule(N=N, max_sweeps=50, rng=rng)
        # [实验控制] 相似记忆的两个起点共享同一随机更新日程
        run_a = d1_run_async(
            weights,
            c1_start_from_memory(memories, 0),
            rng,
            update_schedule=update_schedule,
        )
        run_b = d1_run_async(
            weights,
            c1_start_from_memory(memories, n - 1),
            rng,
            update_schedule=update_schedule,
        )
        # [中介变量] run_a、run_b 分别从这对相似记忆各自出发，看它们会不会收敛到同一个点

        fused += int(np.array_equal(run_a.final_state, run_b.final_state))
        # [判断] 两个终态完全相同 = 融合成了同一个吸引子

        total_error += (
            e1_hamming_distance(run_a.final_state, memories[0])
            + e1_hamming_distance(run_b.final_state, memories[-1])
        ) / 2
        # [观测·数据] 终态相对原记忆的偏移量；即使没融合，也测一下终态相对各自原记忆偏移了多少

    fused_rates.append(fused / trials)
    displaced_errors.append(total_error / trials)

f1_plot_curve(
    distances,
    {
        "融合率": fused_rates,
        "平均偏移量/100": np.asarray(displaced_errors) / 100,
        # [展示] 除以100纯粹是让两条曲线数值量级接近，方便共用一个纵轴看
    },
    xlabel="相似记忆对之间的汉明距离",
    ylabel="比率",
    title="相似记忆：分离到融合",
)

for distance, rate, error in zip(distances, fused_rates, displaced_errors):
    print(f"汉明距离={distance}：融合率={rate:.3f}，平均偏移量={error:.2f}")


**对照论文**：距离 30 时通常都稳定；距离 20 时极小值通常仍分离但位置偏移；距离 10 时经常融合。

**结果分析**：两条记忆越相似，越容易被同一吸引子合并，终态也越容易偏离各自原记忆。


## 机制扩展2：阈值与“不熟悉”状态

**原文定位**：论文 p.2557，讨论统一阈值如何让静默状态成为可能的稳定态。

**问题**：提高阈值后，与任何记忆都不像的输入是否更容易落入全静默状态？

**编码提醒**：论文 $\{0,1\}$ 编码中的 `0000…`，在这里的 $\{-1,+1\}$ 编码中对应全 `-1`。

**配方**：`a1 → b1 → c1/c4 → d1 → e2 → f1`


In [ ]:
rng = np.random.default_rng(109)
N, n, trials = 100, 5, 100
memories = a1_make_independent_memories(n=n, N=N, rng=rng)
weights = b1_make_hebbian_weights(memories)
silent_state = -np.ones(N, dtype=np.int8)
# [编码翻译] 论文 {0,1} 编码里的全 0000... 状态，在 {-1,+1} 编码下对应全 -1
threshold_values = [0.0, 60.0, 70.0, 80.0]
# *[输入] 本实验的自变量：统一阈值
unfamiliar_silent_rates = []
familiar_silent_rates = []

for threshold in threshold_values:
    unfamiliar_silent = 0
    familiar_silent = 0
    for _ in range(trials):
        unfamiliar_state = c4_unfamiliar_state(N=N, rng=rng)
        update_schedule = d0_make_update_schedule(N=N, max_sweeps=50, rng=rng)
        # [实验控制] 同一次熟悉/陌生比较共享随机更新日程
        unfamiliar_run = d1_run_async(
            weights,
            unfamiliar_state,
            rng,
            update_schedule=update_schedule,
            threshold=threshold,
        )
        _, distance = e2_identify_attractor(
            unfamiliar_run.final_state,
            silent_state[None, :],
        )
        # [编码翻译] e2 要求候选记忆是二维数组，silent_state 临时升维成 (1,N)
        unfamiliar_silent += int(distance == 0)
        # [判断] 终态精确等于全静默状态

        familiar_index = int(rng.integers(n))
        familiar_state = c1_start_from_memory(memories, familiar_index)
        familiar_run = d1_run_async(
            weights,
            familiar_state,
            rng,
            update_schedule=update_schedule,
            threshold=threshold,
        )
        _, distance = e2_identify_attractor(
            familiar_run.final_state,
            silent_state[None, :],
        )
        familiar_silent += int(distance == 0)
        # [中介变量] 同样的判定逻辑，但起点换成熟悉输入，作为对照

    unfamiliar_silent_rates.append(unfamiliar_silent / trials)
    familiar_silent_rates.append(familiar_silent / trials)

f1_plot_curve(
    threshold_values,
    {
        "陌生输入": unfamiliar_silent_rates,
        "熟悉输入": familiar_silent_rates,
    },
    xlabel="统一阈值",
    ylabel="静默状态概率",
    title="基于阈值的新颖性识别",
)

import pandas as pd

df = pd.DataFrame({
    "阈值": threshold_values,
    "陌生拒识率": [f"{x:.1%}" for x in unfamiliar_silent_rates],
    "熟悉输入静默率": [f"{x:.1%}" for x in familiar_silent_rates],
})
print(df.to_string(index=False, justify='center'))


**解释**：这是阈值机制的可执行演示。阈值的数值依赖编码和权重是否归一化，不能直接把这里的数值与论文 $\{0,1\}$ 记号逐字等同。

**结果分析**：提高阈值会改变两类输入进入静默状态的概率差异，可形成新颖性识别信号。


## 实验8：严重过载下的熟悉度识别

**原文定位**：论文 p.2558，$N=100,n=500$，论文称记忆过载约 25 倍。

**问题**：当所有记忆都不再是稳定点时，能否通过初始处理速率区分熟悉与陌生输入？

**配方**：`a1 → b1 → c1/c4 → 前 N/2 次异步尝试 → e4 → 配对统计 → 组合图`

**时间换算**：论文设每个神经元的平均尝试速率为 $W$。全网平均事件率为 $NW$，所以时间窗 $1/(2W)$ 内平均发生 $N/2$ 次更新尝试。本实验固定为恰好 $N/2=50$ 次，并按有放回方式随机抽取神经元；熟悉/陌生输入共享同一事件序列。

**论文数据边界**：原文给出了条件、稳定性结论和方向，但没有给熟悉/陌生组的数值均值、误差条或原始样本。因此这里只对照原文确实报告的量，不虚构“论文基准柱”。

In [ ]:
rng = np.random.default_rng(110)
N, n, trials = 100, 500, 100
attempts = N // 2
# *[输入] N=100,n=500、时间窗内50次尝试，逐项对应论文 p.2558

# 1.储存并检查过载后的稳定性----------
memories = a1_make_independent_memories(n=n, N=N, rng=rng)
weights = b1_make_hebbian_weights(memories)

aligned_fields = memories * (memories.astype(float) @ weights.T)
# [观测·数据] 每条记忆、每个神经元的“当前状态 × 局部场”；负值表示该位一更新就会翻转
memory_is_unstable = np.any(aligned_fields < 0, axis=1)
# [判断] 只要一位与局部场方向相反，这条分配记忆就不是固定点

# 2.熟悉/陌生成对测量----------
familiar_counts = []
unfamiliar_counts = []

for _ in range(trials):
    familiar_index = int(rng.integers(n))
    familiar_state = c1_start_from_memory(memories, familiar_index)
    unfamiliar_state = c4_unfamiliar_state(N=N, rng=rng)
    update_order = rng.integers(0, N, size=attempts)
    # [实验控制] 有放回随机事件；同一对输入共享完全相同的50次神经元编号

    familiar_counts.append(
        e4_initial_flip_count(
            weights,
            familiar_state,
            rng,
            attempts=attempts,
            update_order=update_order,
        )
    )
    unfamiliar_counts.append(
        e4_initial_flip_count(
            weights,
            unfamiliar_state,
            rng,
            attempts=attempts,
            update_order=update_order,
        )
    )

familiar_counts = np.asarray(familiar_counts, dtype=float)
unfamiliar_counts = np.asarray(unfamiliar_counts, dtype=float)
paired_difference = unfamiliar_counts - familiar_counts
# [观测·数据] 正值表示该次配对中“陌生输入调整得更快”


def mean_ci95(values: np.ndarray) -> tuple[float, float, float]:
    """返回样本均值与均值的正态近似95%置信区间。"""
    values = np.asarray(values, dtype=float)
    mean = float(np.mean(values))
    margin = 1.96 * float(np.std(values, ddof=1)) / np.sqrt(values.size)
    return mean, mean - margin, mean + margin


familiar_mean, familiar_low, familiar_high = mean_ci95(familiar_counts)
unfamiliar_mean, unfamiliar_low, unfamiliar_high = mean_ci95(unfamiliar_counts)
difference_mean, difference_low, difference_high = mean_ci95(paired_difference)
unfamiliar_faster_rate = float(np.mean(paired_difference > 0))

# 3.画配对数据与置信区间----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(familiar_counts, unfamiliar_counts, alpha=0.55)
limit_low = min(familiar_counts.min(), unfamiliar_counts.min()) - 1
limit_high = max(familiar_counts.max(), unfamiliar_counts.max()) + 1
axes[0].plot([limit_low, limit_high], [limit_low, limit_high], "--", color="gray")
axes[0].set(
    xlim=(limit_low, limit_high),
    ylim=(limit_low, limit_high),
    xlabel="熟悉输入：50次尝试中的调整次数",
    ylabel="陌生输入：50次尝试中的调整次数",
    title="每个点是一组成对试验",
)
# [读图] 点在虚线上方 = 同一次配对中陌生输入调整更多

means = [familiar_mean, unfamiliar_mean, difference_mean]
lower_errors = [
    familiar_mean - familiar_low,
    unfamiliar_mean - unfamiliar_low,
    difference_mean - difference_low,
]
upper_errors = [
    familiar_high - familiar_mean,
    unfamiliar_high - unfamiliar_mean,
    difference_high - difference_mean,
]
axes[1].errorbar(
    [0, 1, 2],
    means,
    yerr=[lower_errors, upper_errors],
    fmt="o",
    capsize=5,
)
axes[1].axhline(0, color="gray", linewidth=1)
axes[1].set_xticks([0, 1, 2], ["熟悉", "陌生", "配对差值\n陌生-熟悉"])
axes[1].set(
    ylabel="平均调整次数（95% CI）",
    title="组均值与配对效应",
)

plt.tight_layout()
plt.show()

# 4.逐项对照论文----------
import pandas as pd

comparison = pd.DataFrame({
    "对照项": [
        "网络与负载",
        "分配记忆稳定性",
        "初始时间窗",
        "熟悉度方向",
        "具体均值与误差",
    ],
    "Hopfield 1982": [
        "N=100,n=500；约25倍过载",
        "全部不稳定",
        "1/(2W) 内的状态调整次数",
        "多数情况下陌生输入更快",
        "未报告",
    ],
    "本次复现": [
        f"N={N},n={n}",
        f"{memory_is_unstable.sum()}/{n} 不稳定（{memory_is_unstable.mean():.1%}）",
        f"固定为 N/2={attempts} 次随机异步尝试",
        f"{unfamiliar_faster_rate:.1%} 的配对中陌生输入调整更多",
        (
            f"熟悉 {familiar_mean:.2f} [{familiar_low:.2f},{familiar_high:.2f}]；"
            f"陌生 {unfamiliar_mean:.2f} [{unfamiliar_low:.2f},{unfamiliar_high:.2f}]"
        ),
    ],
})
display(comparison)

print(
    f"配对差值（陌生-熟悉）={difference_mean:.2f}，"
    f"95% CI=[{difference_low:.2f}, {difference_high:.2f}]"
)

**如何判断是否复现**：先看 500 条分配记忆是否全部不稳定；再看配对差值的 95% 置信区间是否整体大于 0，以及散点是否多数位于对角线上方。满足这两点，才支持论文的定性结论“多数情况下陌生输入处理更快”。

**不能声称什么**：不能说模拟均值“吻合论文数值”，因为论文没有发表那两个均值。这里能复现的是实验条件、指标定义、记忆不稳定性和效应方向。

## 机制扩展3：相关记忆与部分信息补全

**原文定位**：论文 p.2558，式 (11)–(12)。

**问题**：若记忆共享统计相关性，部分输入是否会被补全为符合共同结构的状态？

**配方**：`a3 → b1 → c3 → d1 → e1 → f2`


In [ ]:
rng = np.random.default_rng(111)
N, n, trials = 100, 30, 100
memories, prototype = a3_make_prototype_correlated_memories(
    n=n,
    N=N,
    flip_probability=0.2,
    rng=rng,
)
# [存储] 30条记忆都是同一个 prototype 加20%概率翻转得到的，彼此高度相关

weights = b1_make_hebbian_weights(memories)
before = []
after = []

for _ in range(trials):
    target_index = int(rng.integers(n))
    initial_state = c3_perturb_memory(
        memories[target_index],
        n_flips=30,
        rng=rng,
    )
    # [输入] 从某条具体记忆出发，但故意扰动掉30个比特(约30%)

    before.append(e1_hamming_distance(initial_state, prototype))
    # [观测·数据] 补全前到原型的距离；补全前，起点离"原型"多远

    run = d1_run_async(weights, initial_state, rng)
    after.append(e1_hamming_distance(run.final_state, prototype))
    # [观测·数据] 补全后到原型的距离；补全后，终态离"原型"多远——对照对象是 prototype 不是 target_index，
    # 这是在测"网络有没有把输入拉向共同结构"，不是测能不能召回具体那条记忆

f2_plot_bar(
    ["补全前", "补全后"],
    [float(np.mean(before)), float(np.mean(after))],
    ylabel="到原型的平均汉明距离",
    title="基于相关结构的补全",
)

print(f"到原型的平均汉明距离：补全前={np.mean(before):.2f}，补全后={np.mean(after):.2f}")


**解释**：论文这里给出的是相关矩阵下的机制推演，不是完整统计实验。本节用共同原型构造相关性，展示同一思想；它不是式 (11)–(12) 的逐项精确复刻。

**结果分析**：若补全后平均距离更小，说明网络利用记忆之间的共同原型结构把输入拉向原型。


## 实验9：时间序列记忆

**原文定位**：论文 p.2558，式 (13)，紧邻 *Discussion* 之前。

**问题**：向 Hebb 权重加入从当前记忆指向下一记忆的非对称项，能否让状态按顺序跳转？

**配方**：`a1 → b1+b6 → c1 → d1 → e5 → f2`


In [ ]:
rng = np.random.default_rng(112)
N, n = 100, 4
memories = a1_make_independent_memories(n=n, N=N, rng=rng)
base_weights = b1_make_hebbian_weights(memories)
# [存储] 先算好普通对称权重，序列项在下面单独叠加，方便对比不同强度

strengths = [0.2, 0.5, 1.0]
# *[输入] 本实验的自变量：序列项强度 A（论文式13），扫描三个档位
path_lengths = []
状态名称 = {"fixed": "固定点", "max_sweeps": "达到最大轮数"}
update_schedule = d0_make_update_schedule(N=N, max_sweeps=20, rng=rng)
# [实验控制] 不同序列项强度共享同一随机更新日程

for strength in strengths:
    weights = b6_add_asymmetric_sequence_terms(
        base_weights,
        memories,
        strength=strength,
    )
    # [更新] 每个强度档位都从同一个 base_weights 出发叠加，不会累积上一次循环的序列项

    initial_state = c1_start_from_memory(memories, 0)
    # [输入] 固定从第一条记忆出发，看它能不能被推着往后面的记忆走

    run = d1_run_async(
        weights,
        initial_state,
        rng,
        update_schedule=update_schedule,
        max_sweeps=20,
        record_states=True,
        # [实验控制] 这里必须记录轨迹，因为要看"经过了哪几条记忆"而不只是终态
    )
    path = e5_nearest_memory_path(run.state_history, memories)
    # [观测·数据] 最近记忆路径；把整条轨迹压缩成"依次经过的记忆编号序列"
    path_lengths.append(len(path))
    print(f"A={strength}：最近记忆路径 {path}，状态={状态名称[run.status]}")

f2_plot_bar(
    [str(value) for value in strengths],
    path_lengths,
    ylabel="依次经过的记忆区域数",
    title="非对称序列项的强度",
)



**对照论文**：论文观察到适当强度可使系统在 $V^s$ 附近停留后移向 $V^{s+1}$，但超过四个状态的序列难以生成，即使短序列也不完全可靠。

**结果分析**：序列项增强后，轨迹可能越过更多记忆区域；是否形成稳定长序列仍需看具体路径。


## 实验10：同步与异步更新——固定点还是二周期？

**为什么补这个实验**：Hopfield 1982 使用异步随机更新，并用对称权重下的能量下降保证收敛；如果把全部神经元改为同步更新，同一个能量证明不再成立。

**权威依据**：

- Hopfield 1982：原算法基于异步并行处理；对称权重下，逐个神经元更新使式 (7) 的能量不增加。[[PNAS 论文](https://doi.org/10.1073/pnas.79.8.2554)]
- Goles & Olivos 1980：对称阈值函数做同步整体更新，长期行为只能是不动点或二周期。[[Discrete Mathematics 原论文](https://doi.org/10.1016/0012-365X(80)90121-1)]

**问题**：在同一组对称 Hebb 权重、同一随机初态下，异步与同步更新分别会落入什么终态？二周期是否随负载增加而更常见？

**配方**：`a1 → b1 → c2 → d1/d2 → 终态类型 + _measure_energy → 组合图`

这里扫描 $n=5,15,50$。每个负载生成 20 个独立网络，每个网络取 20 个随机初态，共 400 组成对轨迹。两种更新共享权重和初态；异步模式额外固定自己的随机更新日程。

In [ ]:
rng = np.random.default_rng(113)
N = 100
loads = [5, 15, 50]
networks_per_load = 20
starts_per_network = 20
max_steps = 50
# *[输入] 每个负载 20×20=400 组成对轨迹；不是只挑一个容易出现循环的例子

rows = []
representative = None

for n in loads:
    async_statuses = []
    sync_statuses = []

    for _ in range(networks_per_load):
        memories = a1_make_independent_memories(n=n, N=N, rng=rng)
        weights = b1_make_hebbian_weights(memories)
        # [实验控制] 对称权重是两类经典收敛结论的共同前提

        for _ in range(starts_per_network):
            initial_state = c2_random_state(N=N, rng=rng)
            update_schedule = d0_make_update_schedule(
                N=N,
                max_sweeps=max_steps,
                rng=rng,
            )

            async_run = d1_run_async(
                weights,
                initial_state,
                rng,
                update_schedule=update_schedule,
                max_sweeps=max_steps,
            )
            sync_run = d2_run_sync(
                weights,
                initial_state,
                max_steps=max_steps,
            )
            # [实验控制] 同一权重、同一初态；只替换 d1/d2 更新规则

            async_statuses.append(async_run.status)
            sync_statuses.append(sync_run.status)

            if n == 15 and sync_run.status == "cycle2" and representative is None:
                async_trace = d1_run_async(
                    weights,
                    initial_state,
                    rng,
                    update_schedule=update_schedule,
                    max_sweeps=max_steps,
                    record_energy=True,
                )
                sync_trace = d2_run_sync(
                    weights,
                    initial_state,
                    max_steps=max_steps,
                    record_states=True,
                )
                sync_energy = [
                    _measure_energy(weights, state)
                    for state in sync_trace.state_history
                ]
                representative = (async_trace, sync_trace, sync_energy)
                # [观测·数据] 保存遇到的第一条真实二周期，不手工构造两神经元玩具例子

    total = len(async_statuses)
    rows.append({
        "n": n,
        "配对轨迹数": total,
        "异步固定点": async_statuses.count("fixed") / total,
        "异步未在上限内固定": async_statuses.count("max_sweeps") / total,
        "同步固定点": sync_statuses.count("fixed") / total,
        "同步二周期": sync_statuses.count("cycle2") / total,
        "同步未分类": sync_statuses.count("max_steps") / total,
    })

result = pd.DataFrame(rows)
assert representative is not None
async_trace, sync_trace, sync_energy = representative

# 2.终态类型与真实能量轨迹----------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.arange(len(loads))
width = 0.36

axes[0].bar(
    x - width / 2,
    result["异步固定点"],
    width,
    label="异步：固定点",
)
axes[0].bar(
    x + width / 2,
    result["同步固定点"],
    width,
    label="同步：固定点",
)
axes[0].bar(
    x + width / 2,
    result["同步二周期"],
    width,
    bottom=result["同步固定点"],
    label="同步：二周期",
)
axes[0].set_xticks(x, [str(n) for n in loads])
axes[0].set(
    ylim=(0, 1.05),
    xlabel="存储记忆数 n（N=100）",
    ylabel="轨迹比例",
    title="同一初态，仅替换更新规则",
)
axes[0].legend(loc="lower left", fontsize=8)

axes[1].plot(async_trace.energy_history)
axes[1].set(
    xlabel="已发生的单神经元翻转编号",
    ylabel="Hopfield 能量",
    title="异步：每次翻转后能量不升",
)

tail_start = max(0, len(sync_energy) - 6)
tail_steps = np.arange(tail_start, len(sync_energy))
axes[2].plot(tail_steps, sync_energy[tail_start:], marker="o")
axes[2].set(
    xlabel="同步更新步数",
    ylabel="同一个 Hopfield 能量",
    title="同步：末段放大可见二周期",
)

plt.tight_layout()
plt.show()

# 3.结果数据与定理检查----------
display_frame = result.copy()
for column in ["异步固定点", "异步未在上限内固定", "同步固定点", "同步二周期", "同步未分类"]:
    display_frame[column] = display_frame[column].map(lambda value: f"{value:.1%}")
display(display_frame)

print("异步代表轨迹能量单调不增：", np.all(np.diff(async_trace.energy_history) <= 1e-9))
print("同步代表轨迹状态：", sync_trace.status, f"（{sync_trace.sweeps} 步检出）")
print("同步代表轨迹最后两次能量：", sync_energy[-2:])

**结果分析**：异步列若全部进入固定点，且代表轨迹能量单调不增，就复核了 Hopfield 1982 使用的收敛机制。同步列出现二周期，则复核了 Goles–Olivos 的允许行为；“同步未分类”应为 0，因为对称阈值网络不会产生长度大于 2 的长期周期。

**不要误读速度**：异步横轴是一连串单神经元翻转，同步横轴是全体神经元同时更新一步，二者不是同一计算成本单位。本实验比较的是长期行为类型和能量性质，不宣称哪种实现“更快”。

**选择建议**：若目标是经典 Hopfield 的能量下降与稳定检索，使用异步更新；若硬件必须同步，需要显式接受并检测二周期，不能只看“运行了多少步”就当作收敛。